
# 1. 환경 설정 및 라이브러리 설치



In [1]:
!pip install transformers datasets accelerate evaluate

import os
import torch
import numpy as np
from transformers import BertTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from evaluate import load as load_metric
from google.colab import drive

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


## 1.1 GPU 설정

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

사용 장치: cuda


## 1.2 구글 드라이브 마운트

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


# 2. 하이브리드 토크나이저 로드

In [ ]:
VOCAB_FILE = "/content/drive/MyDrive/bert/hybrid_bert_vocab.txt"

tokenizer = BertTokenizer(vocab_file=VOCAB_FILE, do_lower_case=True)


# 3. 데이터셋 로드 및 전처리


In [ ]:
# Dair-ai-emotion 원본 데이터셋 불러오기
datasets = load_dataset("dair-ai/emotion")

# 전처리 정의
def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# 토큰화 수행
tokenized_datasets = datasets.map(preprocess_function, batched=True)

# 불필요한 콜럼 제거 및 파이토치 포맷 설정
columns_keep = ['input_ids', 'attention_mask', 'label']
columns_erase = [col for col in tokenized_datasets['train'].column_names if col not in columns_keep]
tokenized_datasets = tokenized_datasets.remove_columns(columns_erase)
tokenized_datasets.set_format("torch")

print(" 데이터 전처리 완료.")

 데이터 전처리 완료.


# 4. 모델 로드 및 임베딩 사이즈 조정

In [ ]:
from transformers import BertConfig, AutoModelForSequenceClassification
import torch

# 우리가 만든 모델의 Config
# 변경하면서 최적값 찾아보기
# 꼭 지켜야하는 제약조건
# 임베딩 차원은 헤드 개수와 꼭 맞아떨어져야만 함(hidden_size / hidden layer == 0)
# 관습적인 측면의 코드
# hidden_size * 4 == intermediate_size 보통 그렇게 짠다고 함
config = BertConfig(
    vocab_size = len(tokenizer),
    hidden_size = 512,
    num_hidden_layers = 8,
    num_attention_heads = 8,
    intermediate_size = 2048,
    max_position_embeddings = 256,
    num_labels = 6,
    hidden_dropout_prob = 0.1,
    attention_probs_dropout_prob = 0.1
)

# Config에 맞게 모델 불러오기
model = AutoModelForSequenceClassification.from_config(config)

# 가급적 쿠다를 쓰되 안된다면 cpu로
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Custom BERT model initialized.")
print(f"Structure: L = {config.num_hidden_layers}, H = {config.hidden_size}, Heads = {config.num_attention_heads}")
print(f"Vocab size: {config.vocab_size}")

Custom BERT model initialized.
Structure: L=8, H=512, Heads=8
Vocab size: 15482


# 5. 모델 학습 시작

In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score
import numpy as np

# 평가 지표 정의
# 모델 학습 성능을 평가하기 위한 함수
# 트레이너가 예측한 결과와 정답을 받아서 정확도 계산
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# 몇번 돌려봤지만 우리 모델의 임계점으로 보이는건 에포크 4번
# 이후는 자원 소모도 크고 과적합이 나기도 하므로 딱 5번으로 제한
# 다만 학습률이나 스텝 조절해가면서 정확도 개선
training_args = TrainingArguments(
    output_dir = "./results_from_scratch",
    num_train_epochs = 5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    learning_rate = 5e-5,
    fp16 = True,
    gradient_accumulation_steps = 2,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    # 정확도 기준 가장 좋은 모델을 사용
    metric_for_best_model = "accuracy",
    # 50스텝마다 학습 로그 출력
    logging_steps = 50,
    report_to = "none"
)

# 트레이너 초기화
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["validation"],
    tokenizer = tokenizer,
    compute_metrics = compute_metrics,
)

# 학습
trainer.train()

/tmp/ipython-input-754814813.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 15479}.


Training from scratch started.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.400700,1.375409,0.492500
2,0.867400,0.738547,0.746000
3,0.363300,0.361829,0.871500
4,0.259900,0.300546,0.895000
5,0.211500,0.279500,0.899500


TrainOutput(global_step=1250, training_loss=0.7133340881347656, metrics={'train_runtime': 193.1093, 'train_samples_per_second': 414.273, 'train_steps_per_second': 6.473, 'total_flos': 1565849395200000.0, 'train_loss': 0.7133340881347656, 'epoch': 5.0})

# 5.1 모델 저장하기

In [ ]:
import os

# 기본적으로 90의 정확도를 보이는 모델을 저장
SAVE_PATH = "/content/drive/MyDrive/bert/final_model_90acc"

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)


Model saved to: /content/drive/MyDrive/bert/final_model_90acc


# 6. 파인튜닝 및 모델 정확도 확인

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, BertTokenizer

# 모델 지정 및 6개 감정으로 라벨 인덱스에 매핑
MODEL_PATH = "/content/drive/MyDrive/bert/final_model_90acc"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
labels = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

# 모델과 토크나이저 불러오기
try:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
    model.to(device)
    # 평가 모드로 전환
    model.eval()

# 입력된 텍스트에 대해 6가지 감정 확률을 계산한 뒤
# Softmax값(총합 1.0)으로 반환
def predict_sentiment(text):
    # 전처리하며 텐서 형태로 변환
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    # 텐서 GPU로 이동
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 추론하면서 역전파 비활성화시킴
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 소프트맥스값으로 변환
    probs = F.softmax(logits, dim=-1)
    # 넘파이 배열로 변환
    probs = probs.cpu().numpy()[0]
    # 나온 결과 매핑하기
    results = {labels[i]: float(probs[i]) for i in range(len(labels))}
    return results

# 사용자 입력 받기
USER_INPUT = input("문장을 입력하시오: ")

print(f"\nInput: \"{USER_INPUT}\"")
# 결과 보기 좋게 정리
print("-" * 30)

# 예측 수행하기
predictions = predict_sentiment(USER_INPUT)
# 내림차순 정렬
sorted_pred = sorted(predictions.items(), key=lambda item: item[1], reverse=True)

# 좌로 정렬 후 상세결과는 소수점 세자리수까지
for emotion, score in sorted_pred:
    print(f"{emotion:<10}: {score*100:.3f}%")

print("-" * 30)
# 가장 지배적인 감정 출력
print(f"Prediction: {sorted_pred[0][0].upper()}")


Model loaded successfully.
Enter text to analyze: what have you done this weekend?

Input: "what have you done this weekend?"
------------------------------
anger     : 73.94%
joy       : 13.99%
sadness   : 8.90%
surprise  : 1.85%
fear      : 0.84%
love      : 0.48%
------------------------------
Prediction: ANGER
